# TP - Teoria de Compiladores
### Grupo: 4

###Integrantes
- **Diego Fabrizio Mucha Alvarez - u202317069**
- **Nicole Yessenia Vasquez Tinco - u202322884**
- **Alessandro Daniel Bravo Castillo - u202224501**
- **Linda Libertad Leon Quispe - u202322089**
- **Alessandro Elías Hesse Pulache - u202318347**

In [14]:
!pip install antlr4-python3-runtime==4.13.2

/home/diegomucha/me/programming/university/Compilers Theory/UPC_geoplus/.venv/bin/pip: 2: exec: /home/diegomucha/me/programming/university/Compilers Theory/.venv/bin/python3: not found


In [15]:
!curl -O https://www.antlr.org/download/antlr-4.13.2-complete.jar

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2089k  100 2089k    0     0  8085k      0 --:--:-- --:--:-- --:--:-- 8100k


In [16]:
%%writefile Formas.g4
grammar Formas;

programa: instrucciones EOF;

instrucciones: instruccion*;

instruccion
    : asignacion
    | punto
    | recta
    | triangulo
    | cuadrado
    | pentagono
    | repetir
    | trasladar
    | mostrar
    | circulo
    ;

asignacion: ID IGUAL expr ;

repetir: REPETIR expr VECES LPAREN instruccion+ RPAREN ;

punto: PUNTO ID LPAREN expr COMA expr RPAREN ;

recta
    : RECTA ID LPAREN ID COMA ID RPAREN
    | RECTA ID LPAREN expr COMA expr COMA expr COMA expr RPAREN
    ;

triangulo
    : TRIANGULO ID LPAREN ID COMA ID COMA ID RPAREN
    | TRIANGULO ID LPAREN expr COMA expr COMA expr COMA expr COMA expr COMA expr RPAREN
    ;

cuadrado
    : CUADRADO ID LPAREN ID COMA ID COMA ID COMA ID RPAREN
    | CUADRADO ID LPAREN expr COMA expr COMA expr COMA expr COMA expr COMA expr COMA expr COMA expr RPAREN
    ;

pentagono
    : PENTAGONO ID LPAREN ID COMA ID COMA ID COMA ID COMA ID RPAREN
    | PENTAGONO ID LPAREN expr COMA expr COMA expr COMA expr COMA expr COMA expr COMA expr COMA expr COMA expr COMA expr RPAREN
    ;

circulo
    : CIRCULO ID LPAREN ID COMA expr RPAREN
    | CIRCULO ID LPAREN expr COMA expr RPAREN
    ;

trasladar
    : TRASLADAR LPAREN ID COMA expr COMA expr RPAREN
    ;

mostrar
    : MOSTRAR LPAREN ID RPAREN
    | MOSTRAR LPAREN ID DOT ID LPAREN NUM RPAREN RPAREN 
    ;

expr
    : expr op=(MULT | DIV) expr
    | expr op=(MAS | MENOS) expr
    | NUM
    | ID
    | LPAREN expr RPAREN
    ;

REPETIR   : 'repetir';
VECES     : 'veces';
PUNTO     : 'punto';
RECTA     : 'recta';

TRIANGULO : 'triangulo';
CUADRADO  : 'cuadrado';
PENTAGONO : 'pentagono';
CIRCULO   : 'circulo';
TRASLADAR : 'trasladar';
MOSTRAR   : 'mostrar';

IGUAL : '=';
DOT   : '.';
COMA  : ',';
LPAREN: '(';
RPAREN: ')';

MAS   : '+';
MENOS : '-';
MULT  : '*';
DIV   : '/';

ID  : [a-zA-Z][a-zA-Z0-9_]*;
NUM : [0-9]+ ('.' [0-9]+)?;

WS : [ \t\r\n]+ -> skip;

Overwriting Formas.g4


In [17]:
!java -jar antlr-4.13.2-complete.jar -Dlanguage=Python3 -visitor Formas.g4

In [18]:
%%writefile Visitor.py

import math
if __name__ is not None and "." in __name__:
    from .FormasParser import FormasParser
    from .FormasVisitor import FormasVisitor
else:
    from FormasParser import FormasParser
    from FormasVisitor import FormasVisitor


class Visitor(FormasVisitor):
    def __init__(self):
        self.variables = {}
        self.figuras = {}
        self.html = ""

    ### GET FIGURAS ###
    def getFiguras(self):
        return self.figuras

    ### GET HTML ###
    def getHtml(self):
        return self.html

    ### CALCULAR LINEA NOTABLE
    def calcularLineaNotable(self, triangulo, tipo_linea):
        import math

        puntos = triangulo["puntos"]
        lineas = []
        
        for i in range(3):
            x1, y1 = puntos[i]
            x2, y2 = puntos[(i + 1) % 3]
            x3, y3 = puntos[(i + 2) % 3]

            if tipo_linea == "mediana":
                px = (x2 + x3) / 2
                py = (y2 + y3) / 2

            elif tipo_linea == "altura":
                dx = x3 - x2
                dy = y3 - y2
                t = ((x1 - x2) * dx + (y1 - y2) * dy) / (dx * dx + dy * dy)
                px = x2 + t * dx
                py = y2 + t * dy

            elif tipo_linea == "bisectriz":
                lado1 = math.dist((x1, y1), (x2, y2))
                lado2 = math.dist((x1, y1), (x3, y3))
                px = (lado2 * x2 + lado1 * x3) / (lado1 + lado2)
                py = (lado2 * y2 + lado1 * y3) / (lado1 + lado2)
                
            lineas.append([x1, y1, px, py])

        return lineas

    ### VISIT PROGRAMA ###
    def visitPrograma(self, ctx):
        return self.visit(ctx.instrucciones())

    ### VISIT INSTRUCCIONES ###
    def visitInstrucciones(self, ctx):
        for instruccion in ctx.instruccion():
            self.visit(instruccion)

    ### VISIT REPETIR ###
    def visitRepetir(self, ctx):
        num = self.visit(ctx.expr())
        for i in range(num):
            for instruccion in ctx.instruccion():
                self.visit(instruccion)

    ### VISIT ASIGNACION ###
    def visitAsignacion(self, ctx):
        nombre = ctx.ID().getText()
        valor = self.visit(ctx.expr())
        self.variables[nombre] = valor

    ### VISIT PUNTO ###
    def visitPunto(self, ctx):
        nombre = ctx.ID().getText()
        x = self.visit(ctx.expr(0))
        y = self.visit(ctx.expr(1))
        self.figuras[nombre] = {"tipo": "punto", "x": x, "y": y}

    ### VISIT RECTA ###
    def visitRecta(self, ctx):
        nombre = ctx.ID(0).getText()
        if len(ctx.ID()) == 3:
            p1 = ctx.ID(1).getText()
            p2 = ctx.ID(2).getText()
            punto1 = self.figuras[p1]
            punto2 = self.figuras[p2]
            self.figuras[nombre] = {
                "tipo": "recta",
                "x1": punto1["x"],
                "y1": punto1["y"],
                "x2": punto2["x"],
                "y2": punto2["y"]
            }
        else:
            self.figuras[nombre] = {
                "tipo": "recta",
                "x1": self.visit(ctx.expr(0)),
                "y1": self.visit(ctx.expr(1)),
                "x2": self.visit(ctx.expr(2)),
                "y2": self.visit(ctx.expr(3))
            }

    ### VISIT TRIANGULO ###
    def visitTriangulo(self, ctx):
        nombre = ctx.ID(0).getText()
        if len(ctx.ID()) == 4:
            p1 = self.figuras[ctx.ID(1).getText()]
            p2 = self.figuras[ctx.ID(2).getText()]
            p3 = self.figuras[ctx.ID(3).getText()]
            self.figuras[nombre] = {
                "tipo": "triangulo",
                "puntos": [
                    (p1["x"], p1["y"]),
                    (p2["x"], p2["y"]),
                    (p3["x"], p3["y"])
                ]
            }
        else:
            self.figuras[nombre] = {
                "tipo": "triangulo",
                "puntos": [
                    (self.visit(ctx.expr(0)), self.visit(ctx.expr(1))),
                    (self.visit(ctx.expr(2)), self.visit(ctx.expr(3))),
                    (self.visit(ctx.expr(4)), self.visit(ctx.expr(5)))
                ]
            }
    ### VISIT CUADRADO ###
    def visitCuadrado(self, ctx):
        nombre = ctx.ID(0).getText()
        if len(ctx.ID()) == 5:
            p1 = self.figuras[ctx.ID(1).getText()]
            p2 = self.figuras[ctx.ID(2).getText()]
            p3 = self.figuras[ctx.ID(3).getText()]
            p4 = self.figuras[ctx.ID(4).getText()]
            self.figuras[nombre] = {
                "tipo": "cuadrado",
                "puntos": [
                    (p1["x"], p1["y"]),
                    (p2["x"], p2["y"]),
                    (p3["x"], p3["y"]),
                    (p4["x"], p4["y"])
                ]
            }
        else:
            self.figuras[nombre] = {
                "tipo": "cuadrado",
                "puntos": [
                    (self.visit(ctx.expr(0)), self.visit(ctx.expr(1))),
                    (self.visit(ctx.expr(2)), self.visit(ctx.expr(3))),
                    (self.visit(ctx.expr(4)), self.visit(ctx.expr(5))),
                    (self.visit(ctx.expr(6)), self.visit(ctx.expr(7)))
                ]
            }
    
    ### VISIT CIRCULO ###
    def visitCirculo(self, ctx):
        nombre = ctx.ID(0).getText()
        
        if len(ctx.ID()) == 2:
            punto_id = ctx.ID(1).getText()
            centro = self.figuras[punto_id]
            x = centro["x"]
            y = centro["y"]
            radio = self.visit(ctx.expr(0))
        
        else:
            x = self.visit(ctx.expr(0))
            if ctx.expr(2):
                y = self.visit(ctx.expr(1))
                radio = self.visit(ctx.expr(2))
            else:
                y = self.visit(ctx.expr(1))
                radio = 30
                
        self.figuras[nombre] = {
            "tipo": "circulo",
            "x": x, 
            "y": y, 
            "radio": radio
        }
    
    ### VISIT PENTAGONO ###
    def visitPentagono(self, ctx):
        nombre = ctx.ID(0).getText()

        if len(ctx.ID()) == 6:
            p1 = self.figuras[ctx.ID(1).getText()]
            p2 = self.figuras[ctx.ID(2).getText()]
            p3 = self.figuras[ctx.ID(3).getText()]
            p4 = self.figuras[ctx.ID(4).getText()]
            p5 = self.figuras[ctx.ID(5).getText()]

            self.figuras[nombre] = {
                "tipo": "pentagono",
                "puntos": [
                    (p1["x"], p1["y"]),
                    (p2["x"], p2["y"]),
                    (p3["x"], p3["y"]),
                    (p4["x"], p4["y"]),
                    (p5["x"], p5["y"])
                ]
            }

        else:
            self.figuras[nombre] = {
                "tipo": "pentagono",
                "puntos": [
                    (self.visit(ctx.expr(0)), self.visit(ctx.expr(1))),
                    (self.visit(ctx.expr(2)), self.visit(ctx.expr(3))),
                    (self.visit(ctx.expr(4)), self.visit(ctx.expr(5))),
                    (self.visit(ctx.expr(6)), self.visit(ctx.expr(7))),
                    (self.visit(ctx.expr(8)), self.visit(ctx.expr(9)))
                ]
            }
    ### VISIT TRASLADAR ###
    def visitTrasladar(self, ctx):
        nombre = ctx.ID().getText()
        dx = self.visit(ctx.expr(0))
        dy = self.visit(ctx.expr(1))
        figura = self.figuras[nombre]
        
        if figura["tipo"] in ["punto", "circulo"]:
            figura["x"] += dx
            figura["y"] += dy
        elif figura["tipo"] == "recta":
            figura["x1"] += dx; figura["y1"] += dy
            figura["x2"] += dx; figura["y2"] += dy
        elif figura["tipo"] in ["triangulo", "cuadrado", "pentagono"]:
            nuevos = []
            for x, y in figura["puntos"]:
                nuevos.append((x + dx, y + dy))
            figura["puntos"] = nuevos

    ### VISIT MOSTRAR ###
    def visitMostrar(self, ctx):
        if len(ctx.ID()) == 1:
            nombre = ctx.ID(0).getText()
            figura = self.figuras[nombre]
            self.html += self.generarFigura(nombre, figura)

        elif len(ctx.ID()) == 2:
            nombre = ctx.ID(0).getText()
            figura = self.figuras[nombre]
            linea_notable = ctx.ID(1).getText()
            lineas = self.calcularLineaNotable(figura, linea_notable)
            figura[linea_notable] = lineas
            self.html += self.generarFigura(nombre, figura, int(ctx.NUM().getText()))

    ### GENERAR FIGURA ###
    def generarFigura(self, nombre, figura, id_ln=0):
        codigo = ""
        if figura["tipo"] == "punto":
            x = figura["x"]
            y = figura["y"]
            codigo += f"""
            ctx.beginPath();
            ctx.arc({x}, {y}, 5, 0, 2 * Math.PI);
            ctx.fill();
            ctx.fillText("{nombre}({x},{y})", {x}+10, {y}+10);
            """

        elif figura["tipo"] == "recta":
            x1 = figura["x1"]
            y1 = figura["y1"]
            x2 = figura["x2"]
            y2 = figura["y2"]
            codigo += f"""
            ctx.beginPath();
            ctx.moveTo({x1}, {y1});
            ctx.lineTo({x2}, {y2});
            ctx.stroke();
            ctx.fillText("{nombre}", ({x1}+{x2})/2, ({y1}+{y2})/2 - 12);
            """
            
        elif figura["tipo"] == "triangulo":
            for tipo in ["mediana", "bisectriz", "altura"]:
                if tipo in figura and id_ln > 0:
                    ln = figura[tipo][id_ln - 1]
                    codigo += f"""
                    ctx.beginPath();
                    ctx.moveTo({ln[0]}, {ln[1]});
                    ctx.lineTo({ln[2]}, {ln[3]});
                    ctx.stroke();
                    ctx.fillText("{tipo}", {ln[2]}, {ln[3]});
                    """

            p = figura["puntos"]
            codigo += f"""
            ctx.beginPath();
            ctx.moveTo({p[0][0]}, {p[0][1]});
            ctx.lineTo({p[1][0]}, {p[1][1]});
            ctx.lineTo({p[2][0]}, {p[2][1]});
            ctx.closePath();
            ctx.stroke();
            ctx.fillText(
                "{nombre}",
                ({p[0][0]} + {p[1][0]} + {p[2][0]}) / 3,
                ({p[0][1]} + {p[1][1]} + {p[2][1]}) / 3
            );
            """

        elif figura["tipo"] == "cuadrado":
            p = figura["puntos"]
            codigo += f"""
            ctx.beginPath();
            ctx.moveTo({p[0][0]}, {p[0][1]});
            ctx.lineTo({p[1][0]}, {p[1][1]});
            ctx.lineTo({p[2][0]}, {p[2][1]});
            ctx.lineTo({p[3][0]}, {p[3][1]});
            ctx.closePath();
            ctx.stroke();
            ctx.fillText(
                "{nombre}",
                ({p[0][0]} + {p[1][0]} + {p[2][0]} + {p[3][0]}) / 4,
                ({p[0][1]} + {p[1][1]} + {p[2][1]} + {p[3][1]}) / 4
            );
            """

        elif figura["tipo"] == "circulo":
            x = figura["x"]
            y = figura["y"]
            r = figura["radio"]
            codigo += f"""
            ctx.beginPath();
            ctx.arc({x}, {y}, {r}, 0, 2 * Math.PI);
            ctx.stroke();
            ctx.fillText("{nombre}", {x} + {r} + 5, {y});
            """

        elif figura["tipo"] == "pentagono":
            p = figura["puntos"]

            codigo += f"""
            ctx.beginPath();
            ctx.moveTo({p[0][0]}, {p[0][1]});
            ctx.lineTo({p[1][0]}, {p[1][1]});
            ctx.lineTo({p[2][0]}, {p[2][1]});
            ctx.lineTo({p[3][0]}, {p[3][1]});
            ctx.lineTo({p[4][0]}, {p[4][1]});
            ctx.closePath();
            ctx.stroke();

            ctx.fillText(
                "{nombre}",
                ({p[0][0]} + {p[1][0]} + {p[2][0]} + {p[3][0]} + {p[4][0]}) / 5,
                ({p[0][1]} + {p[1][1]} + {p[2][1]} + {p[3][1]} + {p[4][1]}) / 5
            );
            """
        
        return codigo

    ### VISIT EXPR ###
    def visitExpr(self, ctx):
        if ctx.NUM():
            n = ctx.NUM().getText()
            return float(n) if "." in n else int(n)
            
        if ctx.ID():
            nombre = ctx.ID().getText()
            return self.variables[nombre]
            
        if ctx.getChildCount() == 3:
            if ctx.getChild(0).getText() == "(":
                return self.visit(ctx.expr(0))
                
            izquierda = self.visit(ctx.expr(0))
            derecha = self.visit(ctx.expr(1))
            operador = ctx.getChild(1).getText()
            
            if operador == "+":
                return izquierda + derecha
            elif operador == "-":
                return izquierda - derecha
            elif operador == "*":
                return izquierda * derecha
            elif operador == "/":
                return izquierda / derecha

Overwriting Visitor.py


In [25]:
%%writefile ejemplo1.geo

lado = 120
radio = 45

punto A(120, 120)
punto B(240, 120)
punto C(180, 240)

triangulo T1(A, B, C)

mostrar(A)
mostrar(B)
mostrar(C)
mostrar(T1)

circulo C1(A, 20)
circulo C2(B, 20)
circulo C3(C, 20)

mostrar(C1)
mostrar(C2)
mostrar(C3)

punto D(420, 100)
punto E(560, 100)
punto F(560, 240)
punto G(420, 240)

cuadrado Q1(D, E, F, G)

mostrar(D)
mostrar(E)
mostrar(F)
mostrar(G)

mostrar(Q1)

recta R1(D, F)
recta R2(E, G)

mostrar(R1)
mostrar(R2)

punto H(760, 110)
punto I(860, 170)
punto J(830, 290)
punto K(690, 290)
punto L(660, 170)

pentagono P1(H, I, J, K, L)

mostrar(H)
mostrar(I)
mostrar(J)
mostrar(K)
mostrar(L)

mostrar(P1)

circulo Sol(980, 120)

mostrar(Sol)

repetir 6 veces (
    trasladar(Sol, -15, 15)
    mostrar(Sol)
)

recta Camino1(100, 420, 300, 420)
recta Camino2(300, 420, 500, 500)
recta Camino3(500, 500, 750, 420)

mostrar(Camino1)
mostrar(Camino2)
mostrar(Camino3)

punto X1(880, 420)
punto X2(980, 420)
punto X3(980, 520)
punto X4(880, 520)

cuadrado Casa(X1, X2, X3, X4)

mostrar(Casa)

triangulo Techo(880, 420, 980, 420, 930, 340)

mostrar(Techo)

circulo Ventana(930, 470, 20)

mostrar(Ventana)

Overwriting ejemplo1.geo


In [26]:
%%writefile ejemplo2.geo

punto A1(120, 120)
punto B1(300, 120)
punto C1(210, 300)

triangulo TMed(A1, B1, C1)

mostrar(TMed)
mostrar(TMed.mediana(1))
mostrar(TMed.mediana(2))
mostrar(TMed.mediana(3))

punto A2(450, 100)
punto B2(700, 160)
punto C2(520, 320)

triangulo TAlt(A2, B2, C2)

mostrar(TAlt)
mostrar(TAlt.altura(1))
mostrar(TAlt.altura(2))
mostrar(TAlt.altura(3))

punto A3(850, 120)
punto B3(1080, 180)
punto C3(920, 360)

triangulo TBis(A3, B3, C3)

mostrar(TBis)
mostrar(TBis.bisectriz(1))
mostrar(TBis.bisectriz(2))
mostrar(TBis.bisectriz(3))

Overwriting ejemplo2.geo


In [36]:
from antlr4 import *
from FormasLexer import FormasLexer
from FormasParser import FormasParser
from Visitor import Visitor

input_stream = FileStream('ejemplo1.geo')

lexer = FormasLexer(input_stream)
token_stream = CommonTokenStream(lexer)
parser = FormasParser(token_stream)

tree = parser.programa()

print(tree.toStringTree(recog=parser))

visitor = Visitor()
visitor.visit(tree)

print(visitor.getFiguras())

(programa (instrucciones (instruccion (asignacion lado = (expr 120))) (instruccion (asignacion radio = (expr 45))) (instruccion (punto punto A ( (expr 120) , (expr 120) ))) (instruccion (punto punto B ( (expr 240) , (expr 120) ))) (instruccion (punto punto C ( (expr 180) , (expr 240) ))) (instruccion (triangulo triangulo T1 ( A , B , C ))) (instruccion (mostrar mostrar ( A ))) (instruccion (mostrar mostrar ( B ))) (instruccion (mostrar mostrar ( C ))) (instruccion (mostrar mostrar ( T1 ))) (instruccion (circulo circulo C1 ( A , (expr 20) ))) (instruccion (circulo circulo C2 ( B , (expr 20) ))) (instruccion (circulo circulo C3 ( C , (expr 20) ))) (instruccion (mostrar mostrar ( C1 ))) (instruccion (mostrar mostrar ( C2 ))) (instruccion (mostrar mostrar ( C3 ))) (instruccion (punto punto D ( (expr 420) , (expr 100) ))) (instruccion (punto punto E ( (expr 560) , (expr 100) ))) (instruccion (punto punto F ( (expr 560) , (expr 240) ))) (instruccion (punto punto G ( (expr 420) , (expr 240) )

line 65:19 extraneous input '-' expecting {'(', ID, NUM}
line 90:24 mismatched input ',' expecting ')'


In [37]:
from IPython.display import HTML

html_content = f"""
<div style="padding: 20px; background-color: #f0f0f0; border-radius: 10px;">
  <h2 style="color: #333;">Interfaz de figuras geométricas</h2>

  <canvas id="canvas" width="1200" height="700" style="background-color: white; border: 1px solid #999;"></canvas>

  <script>
    const canvas = document.getElementById("canvas");
    const ctx = canvas.getContext("2d");

    ctx.font = "12px Arial";
    ctx.lineWidth = 2;

    {visitor.getHtml()}
  </script>
</div>
"""

HTML(html_content)

In [32]:
from antlr4 import *
from FormasLexer import FormasLexer
from FormasParser import FormasParser
from Visitor import Visitor

input_stream = FileStream('ejemplo2.geo')

lexer = FormasLexer(input_stream)
token_stream = CommonTokenStream(lexer)
parser = FormasParser(token_stream)

tree = parser.programa()

print(tree.toStringTree(recog=parser))

visitor = Visitor()
visitor.visit(tree)

print(visitor.getFiguras())

(programa (instrucciones (instruccion (punto punto A1 ( (expr 120) , (expr 120) ))) (instruccion (punto punto B1 ( (expr 300) , (expr 120) ))) (instruccion (punto punto C1 ( (expr 210) , (expr 300) ))) (instruccion (triangulo triangulo TMed ( A1 , B1 , C1 ))) (instruccion (mostrar mostrar ( TMed ))) (instruccion (mostrar mostrar ( TMed . mediana ( 1 ) ))) (instruccion (mostrar mostrar ( TMed . mediana ( 2 ) ))) (instruccion (mostrar mostrar ( TMed . mediana ( 3 ) ))) (instruccion (punto punto A2 ( (expr 450) , (expr 100) ))) (instruccion (punto punto B2 ( (expr 700) , (expr 160) ))) (instruccion (punto punto C2 ( (expr 520) , (expr 320) ))) (instruccion (triangulo triangulo TAlt ( A2 , B2 , C2 ))) (instruccion (mostrar mostrar ( TAlt ))) (instruccion (mostrar mostrar ( TAlt . altura ( 1 ) ))) (instruccion (mostrar mostrar ( TAlt . altura ( 2 ) ))) (instruccion (mostrar mostrar ( TAlt . altura ( 3 ) ))) (instruccion (punto punto A3 ( (expr 850) , (expr 120) ))) (instruccion (punto punto

In [24]:
from IPython.display import HTML

html_content = f"""
<div style="padding: 20px; background-color: #f0f0f0; border-radius: 10px;">
  <h2 style="color: #333;">Interfaz de figuras geométricas</h2>

  <canvas id="canvas" width="1200" height="700" style="background-color: white; border: 1px solid #999;"></canvas>

  <script>
    const canvas = document.getElementById("canvas");
    const ctx = canvas.getContext("2d");

    ctx.font = "12px Arial";
    ctx.lineWidth = 2;

    {visitor.getHtml()}
  </script>
</div>
"""

HTML(html_content)